# PromptTemplate

In [3]:
import sys
data = sys.stdin.read().strip().split()
nums = list(map(int, data)) if data else []
print(nums)

[]


In [14]:
!pip install langchain
!pip install -U langchain-core langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.2/504.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 8.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.12
    Uninstalling langchain-1.2.12:
      Successfully uninstalled langchain-1.2.12


In [6]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "Tell me about {peopleN}"
)

prompt_template.format(peopleN = 'zhoukexin')
prompt_template

PromptTemplate(input_variables=['peopleN'], input_types={}, partial_variables={}, template='Tell me about {peopleN}')

# ChatPromptTemplate

In [11]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI bot. Your name is {name}."),
        ("human", "Hello, how are you doing?"),
        ("ai", "I'm doing well, thanks!"),
        ("human", "{user_input}"),
    ]
)

messages = chat_template.format_messages(name="Bob", user_input="What is your name?")

messages
# [SystemMessage(content='You are a helpful AI bot. Your name is Bob.', additional_kwargs={}, response_metadata={}),
#  HumanMessage(content='Hello, how are you doing?', additional_kwargs={}, response_metadata={}),
#  AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
#  HumanMessage(content='What is your name?', additional_kwargs={}, response_metadata={})]

for m in messages:
  print(m)


content='You are a helpful AI bot. Your name is Bob.' additional_kwargs={} response_metadata={}
content='Hello, how are you doing?' additional_kwargs={} response_metadata={}
content="I'm doing well, thanks!" additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='What is your name?' additional_kwargs={} response_metadata={}


In [14]:
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content=(
                "You are a helpful assistant that re-writes the user's text to "
                "sound more upbeat."
            )
        ),
        HumanMessagePromptTemplate.from_template("{text}"),
    ]
)
messages = chat_template.format_messages(text="I don't like eating tasty things") # user Q
messages

[SystemMessage(content="You are a helpful assistant that re-writes the user's text to sound more upbeat.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="I don't like eating tasty things", additional_kwargs={}, response_metadata={})]

# Message Prompts
[different types of messages](https://python.langchain.com/docs/modules/model_io/chat/message_types/)

In [15]:
from langchain_core.prompts import ChatMessagePromptTemplate

prompt = "May the {subject} be with you"

chat_message_prompt = ChatMessagePromptTemplate.from_template(
    role="Jedi", template=prompt
)
chat_message_prompt.format(subject="force") # 指定user

ChatMessage(content='May the force be with you', additional_kwargs={}, response_metadata={}, role='Jedi')

# MessagesPlaceholder

In [28]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
)

human_prompt = "Summarize our conversation so far in {word_count} words."
human_message_template = HumanMessagePromptTemplate.from_template(human_prompt)

chat_prompt = ChatPromptTemplate.from_messages(
    [MessagesPlaceholder(variable_name="conversation"), human_message_template]  #
) # MessagesPlaceholder构建多轮对话的格式

from langchain_core.messages import AIMessage, HumanMessage

human_message = HumanMessage(content="What is the best way to learn programming?")
ai_message = AIMessage(
    content="""\
1. Choose a programming language: Decide on a programming language that you want to learn.

2. Start with the basics: Familiarize yourself with the basic programming concepts such as variables, data types and control structures.

3. Practice, practice, practice: The best way to learn programming is through hands-on experience\
"""
)

# 多轮对话
chat_prompt.format_prompt(
    conversation=[human_message, ai_message], word_count="10"
).to_messages() #

# [历史] 用户问怎么学编程。 【human_message】

# [历史] 我回答了三点建议。 【ai_message】

# [现在] 用户让我用 10 个词总结。 基于上述的历史进行总结

[HumanMessage(content='What is the best way to learn programming?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='1. Choose a programming language: Decide on a programming language that you want to learn.\n\n2. Start with the basics: Familiarize yourself with the basic programming concepts such as variables, data types and control structures.\n\n3. Practice, practice, practice: The best way to learn programming is through hands-on experience', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Summarize our conversation so far in 10 words.', additional_kwargs={}, response_metadata={})]

# Few-shot Prompt

In [20]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
)

examples = [
    {"input": "2+2", "output": "4"},
    {"input": "2+3", "output": "5"},
]

# This is a prompt template used to format each individual example.
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt, ## few-shot prompt=》 case的格式
    examples=examples, ## 正确的case
)

print(few_shot_prompt.format())

Human: 2+2
AI: 4
Human: 2+3
AI: 5


In [29]:
print(few_shot_prompt.format())

print('=====')
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a wondrous wizard of math."), ## 定义作用
        few_shot_prompt, # 正确的case
        ("human", "{input}"), # 新的问题
    ]
)

print(final_prompt.format(input='6+7'))

Human: 2+2
AI: 4
Human: 2+3
AI: 5
=====
System: You are a wondrous wizard of math.
Human: 2+2
AI: 4
Human: 2+3
AI: 5
Human: 6+7
